# History from nsepy

In [ ]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging, logging.config
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

In [ ]:
# get the symbols
from src.nse.nse import get_nse_syms
df_syms = get_nse_syms()

# convert to scrips with is_index = True for Indexes like NIFTY
scrips = df_syms.set_index('symbol')['secType'].eq('IND').to_dict()

In [4]:
# Gets history if they are old

from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm_notebook

from src.nse.nse import get_nse_hist
from src.support import pickle_age

days_old = 1 # set how many days old
hists_file = root / 'data' / 'master'

try:
    hist_age_delta = pickle_age(hists_file)['nse_hists.pkl']
    hist_age = hist_age_delta.days + hist_age_delta.hours/24 + hist_age_delta.minutes/24/60
except KeyError:
    hist_age = days_old + 1 # regenerate

old_hists = hist_age > days_old

dfs = []
hist_days = 365

if old_hists:
    tq_scrips = tqdm_notebook(scrips.items(), bar_format="{desc:<8}{percentage:3.0f}%{r_bar}")
    for k, v in tq_scrips:
        try:
            dfs.append(get_nse_hist(k, v, hist_days))
        except KeyError as e:
            log.error(f"{k} has error {e}")
            pass
        tq_scrips.set_description(f"{k}")
else:
    dfs = [pd.read_pickle(hists_file / 'nse_hists.pkl')]

# assemble the hists
nse_hists = pd.concat(dfs,ignore_index=True)

# store the nse_hists if new
if old_hists:
    data_path = root / 'data' / 'master' / 'nse_hists.pkl'
    nse_hists.to_pickle(data_path)

          0%| 0/195 [00:00<?, ?it/s]

In [ ]:
nse_hists